In [1]:
import os
from pyspark.sql import SparkSession
import pandas as pd
from pyspark.sql.functions import (
    current_timestamp,
    regexp_replace, 
    col, 
    when, 
    coalesce, 
    lit, 
    length
)

In [2]:
# Set JAVA_HOME for PySpark
os.environ['JAVA_HOME'] = '/opt/homebrew/Cellar/openjdk@21/21.0.9/libexec/openjdk.jdk/Contents/Home'

# Stop any existing spark sessions
SparkSession._instantiatedSession = None if SparkSession._instantiatedSession else None
spark = SparkSession.builder \
    .appName("LendingClub") \
    .master("local[2]") \
    .enableHiveSupport() \
    .getOrCreate()


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/02/08 12:31:05 WARN Utils: Your hostname, Himanshus-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 192.168.0.223 instead (on interface en0)
26/02/08 12:31:05 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/08 12:31:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### 1. Cleaning customers.csv

In [4]:
file_path = "../data/raw/customers.csv"

customers_df = spark.read\
        .format("csv")\
        .option("header", "true")\
        .option("inferSchema", "true")\
        .load(file_path)
        
customers_df.show(5)


In [6]:
customers_df_renamed = customers_df.withColumnRenamed("annual_inc", "annual_income")\
                            .withColumnRenamed("addr_state", "address_state")\
                            .withColumnRenamed("zip_code", "address_zipcode")\
                            .withColumnRenamed("country", "address_country")\
                            .withColumnRenamed("tot_hi_cred_lim", "total_high_credit_limit")\
                            .withColumnRenamed("annual_inc_joint", "join_annual_income")

In [ ]:
# Add ingest_date column with current timestamp
customers_df_ingested = customers_df_renamed.withColumn("ingest_date", current_timestamp())
customers_df_ingested.printSchema()


In [ ]:
# Remove duplicate records
customers_df_distinct = customers_df_ingested.distinct()


In [ ]:
# Removing records with null annual_income
customers_df_distinct.createOrReplaceTempView("customers")
customers_income_filtered = spark.sql("select * from customers where annual_income is not null")
customers_income_filtered.show(5)


In [14]:
# Fixing the emp_length column by removing non-numeric characters and converting to integer
customers_income_filtered.createOrReplaceTempView("customers")
spark.sql("select distinct(emp_length) from customers").show()


+----------+
|emp_length|
+----------+
|   5 years|
|   9 years|
|    1 year|
|   2 years|
|   7 years|
|   8 years|
|   4 years|
|   6 years|
|   3 years|
| 10+ years|
|  < 1 year|
| reactors"|
|      NULL|
+----------+



In [15]:
spark.sql("SELECT regexp_replace(emp_length, '[^0-9]', '') AS emp_length FROM customers").show()

+----------+
|emp_length|
+----------+
|         3|
|         2|
|         8|
|        10|
|        10|
|         6|
|         5|
|         3|
|        10|
|         2|
|        10|
|         5|
|        10|
|        10|
|        10|
|        10|
|         1|
|        10|
|         2|
|        10|
+----------+
only showing top 20 rows


In [ ]:
customers_emplength_cleaned = customers_income_filtered.withColumn("emp_length", regexp_replace(col("emp_length"), "[^0-9]",""))
customers_emplength_cleaned.show(5)
customers_emplength_cleaned.printSchema()


In [ ]:
#Replace empty strings with NULL before casting to integer. Replace the nulls with average emp_length
customers_emplength_int = customers_emplength_cleaned.withColumn(
    "emp_length",
    when(col("emp_length") == "", None).otherwise(col("emp_length")).cast('int')
)


In [21]:
customers_emplength_int.createOrReplaceTempView("customers")
avg_emp_length = spark.sql("select floor(avg(emp_length)) as avg_emp_length from customers").collect()
print(avg_emp_length)


[Row(avg_emp_length=6)]


In [22]:
avg_emp_duration = avg_emp_length[0][0]
print(avg_emp_duration)


6


In [ ]:
customers_emplength_replaced = customers_emplength_int.na.fill(avg_emp_duration, subset=['emp_length'])


In [24]:
# Replace incorrect addresses with 'NA'
customers_emplength_replaced.createOrReplaceTempView("customers")
spark.sql("select distinct(address_state) from customers").show(5)
spark.sql("select count(address_state) from customers where length(address_state)>2").show()

+--------------------+
|       address_state|
+--------------------+
|               223xx|
|175 (total projec...|
|                  AZ|
|                  SC|
|financially I mad...|
|                  LA|
|                  MN|
|               499xx|
|yet Capital One n...|
|only to find out ...|
|                  NJ|
|but I would like ...|
|since I was unemp...|
|                  DC|
|and I want to get...|
|       000 per month|
|               553xx|
|we both stand to ...|
|the rent).  John ...|
|Pay Off Credit Cards|
+--------------------+
only showing top 20 rows


In [ ]:
customers_state_cleaned = customers_emplength_replaced.withColumn(
    "address_state",
    when(length(col("address_state"))> 2, "NA").otherwise(col("address_state"))
)
customers_state_cleaned.show(5)
customers_state_cleaned.count()


In [29]:
# Check for duplicate member_id values. Exclude and store the duplicates as bad_data_customers

customers_state_cleaned.createOrReplaceTempView("customers")

spark.sql("""select member_id, count(*) as total 
from customers
group by member_id order by total desc
""").show(5)

+--------------------+-----+
|           member_id|total|
+--------------------+-----+
|e4c167053d5418230...|    5|
|3f87585a20f702838...|    4|
|76b577467eda5bdbc...|    4|
|ad8e5d384dae17e06...|    4|
|1a594f95f63362b33...|    3|
|9240fa4e744ff0846...|    3|
|63b021599856dc21b...|    3|
|066ddaa64bee66dff...|    3|
|e2f7f066058696e76...|    3|
|0480dec0694eacf3f...|    3|
|f8a900a9b2359b5d6...|    3|
|39803eb8abd96d869...|    3|
|937b02d9fd88b9d16...|    3|
|c3b7ec1071de62e44...|    3|
|c0c09bafdbf0655d8...|    3|
|d93573e2883e37904...|    3|
|c80b3e1938d2f7fad...|    3|
|793b618a7de10f813...|    3|
|4cba41b3614fcf461...|    3|
|f42447e25d558b419...|    3|
+--------------------+-----+
only showing top 20 rows


In [36]:
bad_data_customers = spark.sql("""SELECT * 
    FROM customers 
    WHERE member_id IN (
        SELECT member_id
        FROM customers
        GROUP BY member_id
        HAVING COUNT(*) > 1)
    """)

bad_data_customers.show(5)
bad_data_customers.count()

+--------------------+------------+----------+--------------+-------------+-------------+---------------+---------------+-----+---------+-------------------+-----------------------+----------------+------------------+-------------------------+--------------------+
|           member_id|   emp_title|emp_length|home_ownership|annual_income|address_state|address_zipcode|address_country|grade|sub_grade|verification_status|total_high_credit_limit|application_type|join_annual_income|verification_status_joint|         ingest_date|
+--------------------+------------+----------+--------------+-------------+-------------+---------------+---------------+-----+---------+-------------------+-----------------------+----------------+------------------+-------------------------+--------------------+
|0fa3654dfc604c1d4...|     Teacher|        10|      MORTGAGE|      80000.0|           CA|          906xx|            USA|    C|       C2|    Source Verified|               532067.0|       Joint App|       

6411

In [33]:
customers_memberid_cleaned = spark.sql("""SELECT * 
    FROM customers 
    WHERE member_id IN (
        SELECT member_id
        FROM customers
        GROUP BY member_id
        HAVING COUNT(*) = 1)
    """)
customers_memberid_cleaned.show(5)
customers_memberid_cleaned.count()

+--------------------+--------------------+----------+--------------+-------------+-------------+---------------+---------------+-----+---------+-------------------+-----------------------+----------------+------------------+-------------------------+--------------------+
|           member_id|           emp_title|emp_length|home_ownership|annual_income|address_state|address_zipcode|address_country|grade|sub_grade|verification_status|total_high_credit_limit|application_type|join_annual_income|verification_status_joint|         ingest_date|
+--------------------+--------------------+----------+--------------+-------------+-------------+---------------+---------------+-----+---------+-------------------+-----------------------+----------------+------------------+-------------------------+--------------------+
|000a60f6b121d1953...|Customer service ...|         2|      MORTGAGE|      41000.0|           TX|          786xx|            USA|    A|       A2|       Not Verified|               2

2254223

In [28]:
customers_memberid_cleaned.write \
.option("header", True) \
.format("csv") \
.mode("overwrite") \
.option("path", "../data/cleaned/customers.csv") \
.save()

### 2. Cleaning loans.csv

In [38]:
loans_schema = """loan_id string, member_id string, loan_amount float,
    funded_amount float, loan_term_months string, interest_rate float,
    monthly_installment float, issue_date string, loan_status string,
    loan_purpose string, loan_title string"""
    
loans_raw_df = spark.read \
.format("csv") \
.option("header",True) \
.schema(loans_schema) \
.load("../data/raw/loans.csv")

loans_raw_df.show(5)
loans_raw_df.printSchema()


+--------+--------------------+-----------+-------------+----------------+-------------+-------------------+----------+-----------+------------------+--------------------+
| loan_id|           member_id|loan_amount|funded_amount|loan_term_months|interest_rate|monthly_installment|issue_date|loan_status|      loan_purpose|          loan_title|
+--------+--------------------+-----------+-------------+----------------+-------------+-------------------+----------+-----------+------------------+--------------------+
|56633077|b59d80da191f5b573...|     3000.0|       3000.0|       36 months|         7.89|              93.86|  Aug-2015| Fully Paid|       credit_card|Credit card refin...|
|55927518|202d9f56ecb7c3bc9...|    15600.0|      15600.0|       36 months|         7.89|             488.06|  Aug-2015| Fully Paid|       credit_card|Credit card refin...|
|56473345|e5a140c0922b554b9...|    20000.0|      20000.0|       36 months|         9.17|             637.58|  Aug-2015| Fully Paid|debt_cons

26/02/08 13:03:20 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: loan_id, member_id, loan_amnt, funded_amnt, term, int_rate, installment, issue_d, loan_status, purpose, title
 Schema: loan_id, member_id, loan_amount, funded_amount, loan_term_months, interest_rate, monthly_installment, issue_date, loan_status, loan_purpose, loan_title
Expected: loan_amount but found: loan_amnt
CSV file: file:///Users/himanshuhegde/Desktop/Lending_club/data/raw/loans.csv


In [40]:
# Add ingest_date column with current timestamp
loans_df_ingestd = loans_raw_df.withColumn("ingest_date", current_timestamp())


In [41]:
# Check for columns with null values
loans_df_ingestd.createOrReplaceTempView("loans")
spark.sql("select * from loans where loan_amount is null").show(5)


26/02/08 13:04:13 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: loan_id, member_id, loan_amnt, funded_amnt, term, int_rate, installment, issue_d, loan_status, purpose, title
 Schema: loan_id, member_id, loan_amount, funded_amount, loan_term_months, interest_rate, monthly_installment, issue_date, loan_status, loan_purpose, loan_title
Expected: loan_amount but found: loan_amnt
CSV file: file:///Users/himanshuhegde/Desktop/Lending_club/data/raw/loans.csv


+--------------------+--------------------+-----------+-------------+----------------+-------------+-------------------+----------+-----------+------------+----------+--------------------+
|             loan_id|           member_id|loan_amount|funded_amount|loan_term_months|interest_rate|monthly_installment|issue_date|loan_status|loan_purpose|loan_title|         ingest_date|
+--------------------+--------------------+-----------+-------------+----------------+-------------+-------------------+----------+-----------+------------+----------+--------------------+
|Total amount fund...|e3b0c44298fc1c149...|       NULL|         NULL|            NULL|         NULL|               NULL|      NULL|       NULL|        NULL|      NULL|2026-02-08 13:04:...|
|Total amount fund...|e3b0c44298fc1c149...|       NULL|         NULL|            NULL|         NULL|               NULL|      NULL|       NULL|        NULL|      NULL|2026-02-08 13:04:...|
|Total amount fund...|e3b0c44298fc1c149...|       NULL|

In [42]:
columns_with_na = ["loan_amount", "funded_amount", "loan_term_months", "interest_rate", "monthly_installment", "issue_date", "loan_status", "loan_purpose", "loan_title"]
loans_filtered_df = loans_df_ingestd.na.drop(subset = columns_with_na)



In [43]:
# Convert loan_term_months from string to integer and convert the column to loan_term_years

loans_term_modified_df = loans_filtered_df.withColumn("loan_term_months", 
    (regexp_replace(col("loan_term_months"), " months", "") \
    .cast("int") / 12).cast("int")).withColumnRenamed("loan_term_months","loan_term_years")

loans_term_modified_df.show(5)
loans_term_modified_df.printSchema()


+--------+--------------------+-----------+-------------+---------------+-------------+-------------------+----------+-----------+------------------+--------------------+--------------------+
| loan_id|           member_id|loan_amount|funded_amount|loan_term_years|interest_rate|monthly_installment|issue_date|loan_status|      loan_purpose|          loan_title|         ingest_date|
+--------+--------------------+-----------+-------------+---------------+-------------+-------------------+----------+-----------+------------------+--------------------+--------------------+
|56633077|b59d80da191f5b573...|     3000.0|       3000.0|              3|         7.89|              93.86|  Aug-2015| Fully Paid|       credit_card|Credit card refin...|2026-02-08 13:06:...|
|55927518|202d9f56ecb7c3bc9...|    15600.0|      15600.0|              3|         7.89|             488.06|  Aug-2015| Fully Paid|       credit_card|Credit card refin...|2026-02-08 13:06:...|
|56473345|e5a140c0922b554b9...|    20000

26/02/08 13:06:38 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: loan_id, member_id, loan_amnt, funded_amnt, term, int_rate, installment, issue_d, loan_status, purpose, title
 Schema: loan_id, member_id, loan_amount, funded_amount, loan_term_months, interest_rate, monthly_installment, issue_date, loan_status, loan_purpose, loan_title
Expected: loan_amount but found: loan_amnt
CSV file: file:///Users/himanshuhegde/Desktop/Lending_club/data/raw/loans.csv


In [45]:
# Analyze and consolidate the loan_purpose column. Valid categories and "others"
loans_term_modified_df.createOrReplaceTempView("loans")
spark.sql("select loan_purpose, count(*) as total from loans group by loan_purpose order by total desc").show()


26/02/08 13:11:24 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: loan_amnt, funded_amnt, term, int_rate, installment, issue_d, loan_status, purpose, title
 Schema: loan_amount, funded_amount, loan_term_months, interest_rate, monthly_installment, issue_date, loan_status, loan_purpose, loan_title
Expected: loan_amount but found: loan_amnt
CSV file: file:///Users/himanshuhegde/Desktop/Lending_club/data/raw/loans.csv


+--------------------+-------+
|        loan_purpose|  total|
+--------------------+-------+
|  debt_consolidation|1263422|
|         credit_card| 511459|
|    home_improvement| 149141|
|               other| 138623|
|      major_purchase|  50050|
|             medical|  27247|
|      small_business|  24416|
|                 car|  23815|
|            vacation|  15373|
|              moving|  15297|
|               house|  14051|
|             wedding|   2351|
|    renewable_energy|   1433|
|         educational|    410|
|and also pay off ...|      1|
|        guaranteed!"|      1|
|and if they are a...|      1|
|never had any tro...|      1|
|<br/><br/>Lending...|      1|
|Bank of America c...|      1|
+--------------------+-------+
only showing top 20 rows


In [47]:
loan_purposes_valid = ["debt_consolidation", "credit_card", "home_improvement", 
                       "other", "major_purchase", "medical", "small_business", 
                       "car", "vacation", "moving", "house", "wedding", 
                       "renewable_energy", "educational"]

loans_purpose_modified = loans_term_modified_df.withColumn(
    "loan_purpose", when(col("loan_purpose").isin(loan_purposes_valid), col("loan_purpose")).otherwise("other"))

loans_purpose_modified.createOrReplaceTempView("loans")
spark.sql("select loan_purpose, count(*) as total from loans group by loan_purpose order by total desc").show()


26/02/08 13:14:18 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: loan_amnt, funded_amnt, term, int_rate, installment, issue_d, loan_status, purpose, title
 Schema: loan_amount, funded_amount, loan_term_months, interest_rate, monthly_installment, issue_date, loan_status, loan_purpose, loan_title
Expected: loan_amount but found: loan_amnt
CSV file: file:///Users/himanshuhegde/Desktop/Lending_club/data/raw/loans.csv


+------------------+-------+
|      loan_purpose|  total|
+------------------+-------+
|debt_consolidation|1263422|
|       credit_card| 511459|
|  home_improvement| 149141|
|             other| 138876|
|    major_purchase|  50050|
|           medical|  27247|
|    small_business|  24416|
|               car|  23815|
|          vacation|  15373|
|            moving|  15297|
|             house|  14051|
|           wedding|   2351|
|  renewable_energy|   1433|
|       educational|    410|
+------------------+-------+



In [51]:
# Remove records with duplicate member_id values. Exclude and store the duplicates as bad_data_loans
loans_purpose_modified.createOrReplaceTempView("loans")

spark.sql("""select member_id, count(*) as total 
from loans
group by member_id order by total desc
""").show(5)


26/02/08 13:18:21 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: member_id, loan_amnt, funded_amnt, term, int_rate, installment, issue_d, loan_status, purpose, title
 Schema: member_id, loan_amount, funded_amount, loan_term_months, interest_rate, monthly_installment, issue_date, loan_status, loan_purpose, loan_title
Expected: loan_amount but found: loan_amnt
CSV file: file:///Users/himanshuhegde/Desktop/Lending_club/data/raw/loans.csv


+--------------------+-----+
|           member_id|total|
+--------------------+-----+
|e4c167053d5418230...|    5|
|76b577467eda5bdbc...|    4|
|ad8e5d384dae17e06...|    4|
|4cba41b3614fcf461...|    3|
|bca141343d9405a9a...|    3|
+--------------------+-----+
only showing top 5 rows


In [48]:
loans_purpose_modified.write \
.option("header", True) \
.format("csv") \
.mode("overwrite") \
.option("path", "../data/cleaned/loans.csv")\
.save()

26/02/08 13:14:48 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: loan_id, member_id, loan_amnt, funded_amnt, term, int_rate, installment, issue_d, loan_status, purpose, title
 Schema: loan_id, member_id, loan_amount, funded_amount, loan_term_months, interest_rate, monthly_installment, issue_date, loan_status, loan_purpose, loan_title
Expected: loan_amount but found: loan_amnt
CSV file: file:///Users/himanshuhegde/Desktop/Lending_club/data/raw/loans.csv


### 3. Cleaning repayments.csv

In [93]:
loans_repay_raw_df = spark.read \
.format("csv") \
.option("header",True) \
.option("inferSchema", True) \
.load("../data/raw/repayments.csv")

loans_repay_raw_df.show(5)
loans_repay_raw_df.printSchema()

+---------+---------------+-------------+------------------+-----------+---------------+------------+------------+
|  loan_id|total_rec_prncp|total_rec_int|total_rec_late_fee|total_pymnt|last_pymnt_amnt|last_pymnt_d|next_pymnt_d|
+---------+---------------+-------------+------------------+-----------+---------------+------------+------------+
|141581221|        1055.81|       2591.7|               0.0|    3647.51|         709.23|    Mar-2019|    Apr-2019|
|141506948|        1252.75|       306.04|               0.0|    1558.79|         312.63|    Mar-2019|    Apr-2019|
|141357400|         626.37|       354.96|               0.0|     981.33|         197.27|    Mar-2019|    Apr-2019|
|139445427|        1118.16|       297.36|               0.0|    1415.52|         283.95|    Mar-2019|    Apr-2019|
|141407409|        1169.72|       3605.3|               0.0|    4775.02|          964.9|    Mar-2019|    Apr-2019|
+---------+---------------+-------------+------------------+-----------+--------

In [94]:
loans_repay_schema = """loan_id string, total_principal_received float, 
    total_interest_received float, total_late_fee_received float, 
    total_payment_received float, last_payment_amount float, 
    last_payment_date string, next_payment_date string"""

loans_repay_raw_df = spark.read \
.format("csv") \
.option("header",True) \
.schema(loans_repay_schema) \
.load("../data/raw/repayments.csv")

loans_repay_raw_df.printSchema()

root
 |-- loan_id: string (nullable = true)
 |-- total_principal_received: float (nullable = true)
 |-- total_interest_received: float (nullable = true)
 |-- total_late_fee_received: float (nullable = true)
 |-- total_payment_received: float (nullable = true)
 |-- last_payment_amount: float (nullable = true)
 |-- last_payment_date: string (nullable = true)
 |-- next_payment_date: string (nullable = true)



In [95]:
# Add ingest_date column with current timestamp
loans_repay_df_ingestd = loans_repay_raw_df.withColumn("ingest_date", current_timestamp())

loans_repay_df_ingestd.printSchema()

root
 |-- loan_id: string (nullable = true)
 |-- total_principal_received: float (nullable = true)
 |-- total_interest_received: float (nullable = true)
 |-- total_late_fee_received: float (nullable = true)
 |-- total_payment_received: float (nullable = true)
 |-- last_payment_amount: float (nullable = true)
 |-- last_payment_date: string (nullable = true)
 |-- next_payment_date: string (nullable = true)
 |-- ingest_date: timestamp (nullable = false)



In [96]:
loans_repay_df_ingestd.createOrReplaceTempView("loan_repayments")

In [97]:
spark.sql("select count(*) from loan_repayments where total_principal_received is null").show()

26/02/08 22:22:52 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: total_rec_prncp
 Schema: total_principal_received
Expected: total_principal_received but found: total_rec_prncp
CSV file: file:///Users/himanshuhegde/Desktop/Lending_club/data/raw/repayments.csv


+--------+
|count(1)|
+--------+
|      69|
+--------+



In [98]:
columns_to_check = ["total_principal_received", "total_interest_received", "total_late_fee_received", "total_payment_received"]

In [99]:
loans_repay_filtered_df = loans_repay_df_ingestd.na.drop(subset=columns_to_check)

In [100]:
loans_repay_filtered_df.createOrReplaceTempView("loan_repayments")

In [101]:
spark.sql("select count(*) from loan_repayments where total_payment_received = 0.0").show()

26/02/08 22:22:59 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: total_rec_prncp, total_rec_int, total_rec_late_fee, total_pymnt
 Schema: total_principal_received, total_interest_received, total_late_fee_received, total_payment_received
Expected: total_principal_received but found: total_rec_prncp
CSV file: file:///Users/himanshuhegde/Desktop/Lending_club/data/raw/repayments.csv


+--------+
|count(1)|
+--------+
|    1053|
+--------+



In [102]:
spark.sql("select count(*) from loan_repayments where total_payment_received = 0.0 and total_principal_received != 0.0").show()

26/02/08 22:23:03 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: total_rec_prncp, total_rec_int, total_rec_late_fee, total_pymnt
 Schema: total_principal_received, total_interest_received, total_late_fee_received, total_payment_received
Expected: total_principal_received but found: total_rec_prncp
CSV file: file:///Users/himanshuhegde/Desktop/Lending_club/data/raw/repayments.csv


+--------+
|count(1)|
+--------+
|     103|
+--------+



In [103]:
loans_payments_fixed_df = loans_repay_filtered_df.withColumn(
   "total_payment_received",
    when(
        (col("total_principal_received") != 0.0) &
        (col("total_payment_received") == 0.0),
        col("total_principal_received") + col("total_interest_received") + col("total_late_fee_received")
    ).otherwise(col("total_payment_received"))
)

In [104]:
loans_payments_fixed_df.show()

+---------+------------------------+-----------------------+-----------------------+----------------------+-------------------+-----------------+-----------------+--------------------+
|  loan_id|total_principal_received|total_interest_received|total_late_fee_received|total_payment_received|last_payment_amount|last_payment_date|next_payment_date|         ingest_date|
+---------+------------------------+-----------------------+-----------------------+----------------------+-------------------+-----------------+-----------------+--------------------+
|141581221|                 1055.81|                 2591.7|                    0.0|               3647.51|             709.23|         Mar-2019|         Apr-2019|2026-02-08 22:23:...|
|141506948|                 1252.75|                 306.04|                    0.0|               1558.79|             312.63|         Mar-2019|         Apr-2019|2026-02-08 22:23:...|
|141357400|                  626.37|                 354.96|               

26/02/08 22:23:10 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: loan_id, total_rec_prncp, total_rec_int, total_rec_late_fee, total_pymnt, last_pymnt_amnt, last_pymnt_d, next_pymnt_d
 Schema: loan_id, total_principal_received, total_interest_received, total_late_fee_received, total_payment_received, last_payment_amount, last_payment_date, next_payment_date
Expected: total_principal_received but found: total_rec_prncp
CSV file: file:///Users/himanshuhegde/Desktop/Lending_club/data/raw/repayments.csv


In [105]:
loans_payments_fixed2_df = loans_payments_fixed_df.filter("total_payment_received != 0.0")

In [106]:
loans_payments_fixed2_df.createOrReplaceTempView("loan_repayments")

In [107]:
spark.sql("select last_payment_date, count(*) as total from loan_repayments group by last_payment_date order by total").show(178)

26/02/08 22:23:14 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: total_rec_prncp, total_rec_int, total_rec_late_fee, total_pymnt, last_pymnt_d
 Schema: total_principal_received, total_interest_received, total_late_fee_received, total_payment_received, last_payment_date
Expected: total_principal_received but found: total_rec_prncp
CSV file: file:///Users/himanshuhegde/Desktop/Lending_club/data/raw/repayments.csv


+------------------+------+
| last_payment_date| total|
+------------------+------+
|361.10519999940004|     1|
|           4336.19|     1|
|14.410300001600001|     1|
|           3890.64|     1|
|           1794.85|     1|
|            600.15|     1|
|            2590.8|     1|
|              1.23|     1|
|              22.0|     1|
|              84.3|     1|
|           3572.93|     1|
|            138.05|     1|
|             410.0|     1|
|            883.07|     1|
|            499.58|     1|
|           1163.37|     1|
|            623.57|     1|
|            354.73|     1|
|             45.78|     1|
|           5019.44|     1|
|           1337.01|     1|
|           2392.02|     1|
|            929.22|     1|
|            624.78|     1|
|           1125.12|     1|
|           1357.25|     1|
|           2046.31|     1|
|            550.11|     1|
|            7908.4|     1|
|              3.64|     1|
|           1071.04|     1|
|           1330.91|     1|
|             134.2|

In [108]:
loans_payments_ldate_fixed_df = loans_payments_fixed2_df.withColumn(
    "next_payment_date",
    when(col("last_payment_date").rlike(r'^\d+(\.\d+)?$'), None)\
        .otherwise(col("last_payment_date"))
)

In [109]:
loans_payments_ndate_fixed_df = loans_payments_ldate_fixed_df.withColumn(
  "last_payment_date",
   when(
       (col("next_payment_date").rlike(r'^\d+(\.\d+)?$')),
       None
       ).otherwise(col("next_payment_date"))
)

In [110]:
loans_payments_ndate_fixed_df.write \
.option("header", True) \
.format("csv") \
.mode("overwrite") \
.option("path", "../data/cleaned/repayments.csv")\
.save()

26/02/08 22:23:24 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: loan_id, total_rec_prncp, total_rec_int, total_rec_late_fee, total_pymnt, last_pymnt_amnt, last_pymnt_d
 Schema: loan_id, total_principal_received, total_interest_received, total_late_fee_received, total_payment_received, last_payment_amount, last_payment_date
Expected: total_principal_received but found: total_rec_prncp
CSV file: file:///Users/himanshuhegde/Desktop/Lending_club/data/raw/repayments.csv


### 4. Cleaning delinquencies.csv

In [111]:
loans_def_raw_df = spark.read \
.format("csv") \
.option("header",True) \
.option("inferSchema", True) \
.load("../data/raw/delinquencies.csv")

loans_def_raw_df.show(5)
loans_def_raw_df.printSchema()

+--------------------+-----------+-----------+-------+--------------------+--------------+------------------+----------------------+----------------------+
|           member_id|delinq_2yrs|delinq_amnt|pub_rec|pub_rec_bankruptcies|inq_last_6mths|total_rec_late_fee|mths_since_last_delinq|mths_since_last_record|
+--------------------+-----------+-----------+-------+--------------------+--------------+------------------+----------------------+----------------------+
|9cb79aa7323e81be1...|        2.0|        0.0|    0.0|                 0.0|           0.0|               0.0|                  11.0|                  NULL|
|0dd2bbc517e3c8f9e...|        0.0|        0.0|    1.0|                 1.0|           3.0|               0.0|                  NULL|                 115.0|
|458458599d3df3bfc...|        0.0|        0.0|    1.0|                 1.0|           1.0|               0.0|                  NULL|                  76.0|
|05ea141ec28b5c7f7...|        0.0|        0.0|    0.0|          

In [112]:
loan_defaulters_schema = """member_id string, delinq_2yrs float, delinq_amnt float, 
pub_rec float, pub_rec_bankruptcies float, inq_last_6mths float, 
total_rec_late_fee float, mths_since_last_delinq float, 
mths_since_last_record float"""

loans_def_raw_df = spark.read \
.format("csv") \
.option("header",True) \
.schema(loan_defaulters_schema) \
.load("../data/raw/delinquencies.csv")


In [113]:
# Check for duplicate member_id values. Exclude and store the duplicates as bad_data_customers

loans_def_raw_df.createOrReplaceTempView("delinquencies")

spark.sql("""select member_id, count(*) as total 
from delinquencies
group by member_id order by total desc
""").show(5)

+--------------------+-----+
|           member_id|total|
+--------------------+-----+
|e3b0c44298fc1c149...|   33|
|e4c167053d5418230...|    5|
|3f87585a20f702838...|    4|
|76b577467eda5bdbc...|    4|
|ad8e5d384dae17e06...|    4|
+--------------------+-----+
only showing top 5 rows


In [114]:
bad_data_delinquencies = spark.sql("""SELECT * 
    FROM delinquencies 
    WHERE member_id IN (
        SELECT member_id
        FROM delinquencies
        GROUP BY member_id
        HAVING COUNT(*) > 1)
    """)

bad_data_delinquencies.show(5)
bad_data_delinquencies.count()

+--------------------+-----------+-----------+-------+--------------------+--------------+------------------+----------------------+----------------------+
|           member_id|delinq_2yrs|delinq_amnt|pub_rec|pub_rec_bankruptcies|inq_last_6mths|total_rec_late_fee|mths_since_last_delinq|mths_since_last_record|
+--------------------+-----------+-----------+-------+--------------------+--------------+------------------+----------------------+----------------------+
|0046c736dca5af995...|        1.0|        0.0|    0.0|                 0.0|           0.0|               0.0|                  12.0|                  NULL|
|0046c736dca5af995...|        0.0|        0.0|    0.0|                 0.0|           1.0|               0.0|                  25.0|                  NULL|
|0183c98620a713219...|        1.0|        0.0|    0.0|                 0.0|           1.0|               0.0|                   8.0|                  NULL|
|0183c98620a713219...|        0.0|        0.0|    0.0|          

6506

In [115]:
delinquencies_memberid_cleaned = spark.sql("""SELECT * 
    FROM delinquencies 
    WHERE member_id IN (
        SELECT member_id
        FROM delinquencies
        GROUP BY member_id
        HAVING COUNT(*) = 1)
    """)

delinquencies_memberid_cleaned.show(5)
delinquencies_memberid_cleaned.count()

+--------------------+-----------+-----------+-------+--------------------+--------------+------------------+----------------------+----------------------+
|           member_id|delinq_2yrs|delinq_amnt|pub_rec|pub_rec_bankruptcies|inq_last_6mths|total_rec_late_fee|mths_since_last_delinq|mths_since_last_record|
+--------------------+-----------+-----------+-------+--------------------+--------------+------------------+----------------------+----------------------+
|0000036e9afe019a6...|        2.0|        0.0|    0.0|                 0.0|           3.0|               0.0|                  15.0|                  NULL|
|00001c0c295167b3e...|        0.0|        0.0|    0.0|                 0.0|           0.0|               0.0|                  NULL|                  NULL|
|00006cfe783695214...|        0.0|        0.0|    1.0|                 1.0|           0.0|             21.44|                  58.0|                  84.0|
|000128ed80bee9d2e...|        1.0|        0.0|    0.0|          

2254195

In [116]:
# converting delinq_2yrs column to integer and replacing nulls with 0
loans_def_processed_df = delinquencies_memberid_cleaned.withColumn(
    "delinq_2yrs", col("delinq_2yrs").cast("integer")).fillna(0, subset = ["delinq_2yrs"])


In [117]:
loans_def_processed_df.createOrReplaceTempView("loan_defaulters")
spark.sql("""select count(*)
          from loan_defaulters 
          where delinq_2yrs is null""").show()


+--------+
|count(1)|
+--------+
|       0|
+--------+



In [118]:
loans_def_delinq_df = spark.sql("""
    SELECT member_id, delinq_2yrs, delinq_amnt, int(mths_since_last_delinq) 
    FROM loan_defaulters 
    WHERE delinq_2yrs > 0 OR mths_since_last_delinq > 0
    """)

loans_def_delinq_df.show(5)


+--------------------+-----------+-----------+----------------------+
|           member_id|delinq_2yrs|delinq_amnt|mths_since_last_delinq|
+--------------------+-----------+-----------+----------------------+
|0000036e9afe019a6...|          2|        0.0|                    15|
|00006cfe783695214...|          0|        0.0|                    58|
|000077b348e7220f6...|          1|        0.0|                    18|
|00012644aa138932f...|          2|        0.0|                    18|
|000128ed80bee9d2e...|          1|        0.0|                    23|
+--------------------+-----------+-----------+----------------------+
only showing top 5 rows


In [119]:
loans_def_delinq_df.write \
.option("header", True) \
.format("csv") \
.mode("overwrite") \
.option("path", "../data/cleaned/delinquencies.csv") \
.save()

Creating Defaulters details

In [120]:
loans_def_p_pub_rec_df = loans_def_processed_df.withColumn(
    "pub_rec", col("pub_rec").cast("integer")).fillna(0, subset = ["pub_rec"])


In [121]:
loans_def_p_pub_rec_bankruptcies_df = loans_def_p_pub_rec_df.withColumn(
    "pub_rec_bankruptcies", col("pub_rec_bankruptcies").cast("integer")).fillna(0, subset = ["pub_rec_bankruptcies"])


In [122]:
loans_def_p_inq_last_6mths_df = loans_def_p_pub_rec_bankruptcies_df.withColumn(
    "inq_last_6mths", col("inq_last_6mths").cast("integer")).fillna(0, subset = ["inq_last_6mths"])


In [123]:
loans_def_p_inq_last_6mths_df.createOrReplaceTempView("loan_defaulters")


In [124]:
loans_def_detail_records_enq_df = spark.sql("""
    SELECT member_id, pub_rec, pub_rec_bankruptcies, inq_last_6mths 
    FROM loan_defaulters
    """)
loans_def_detail_records_enq_df.show(5)
loans_def_detail_records_enq_df.count()

+--------------------+-------+--------------------+--------------+
|           member_id|pub_rec|pub_rec_bankruptcies|inq_last_6mths|
+--------------------+-------+--------------------+--------------+
|0000036e9afe019a6...|      0|                   0|             3|
|00001c0c295167b3e...|      0|                   0|             0|
|00006cfe783695214...|      1|                   1|             0|
|000128ed80bee9d2e...|      0|                   0|             0|
|00012f158a95b9c9f...|      0|                   0|             0|
+--------------------+-------+--------------------+--------------+
only showing top 5 rows


2254195

In [125]:
loans_def_detail_records_enq_df.write \
.option("header", True) \
.format("csv") \
.mode("overwrite") \
.option("path", "../data/cleaned/loans_del_detail_records_enq_df.csv") \
.save()

### 5. Creating permanent tables on the cleaned datasets

In [ ]:
spark.sql("create database lending_club")

In [147]:
spark.sql("drop table if exists lending_club.customers")
spark.sql("drop table if exists lending_club.loans")
spark.sql("drop table if exists lending_club.repayments")
spark.sql("drop table if exists lending_club.delinquencies")
spark.sql("drop table if exists lending_club.delinquencies_rec_enq")


DataFrame[]

In [149]:
spark.sql("""CREATE EXTERNAL TABLE lending_club.customers(
member_id STRING, emp_title STRING, emp_length INT, 
home_ownership string, annual_income float, address_state string, address_zipcode string, address_country string, grade string, 
sub_grade string, verification_status string, total_high_credit_limit float, application_type string, join_annual_income float, 
verification_status_joint string, ingest_date timestamp)
using csv location  '/Users/himanshuhegde/Desktop/Lending_club/data/cleaned/customers.csv'
options('header'='true', 'inferSchema'='false')
""")

spark.sql("select * from lending_club.customers").show(5)


+--------------------+--------------------+----------+--------------+-------------+-------------+---------------+---------------+-----+---------+-------------------+-----------------------+----------------+------------------+-------------------------+--------------------+
|           member_id|           emp_title|emp_length|home_ownership|annual_income|address_state|address_zipcode|address_country|grade|sub_grade|verification_status|total_high_credit_limit|application_type|join_annual_income|verification_status_joint|         ingest_date|
+--------------------+--------------------+----------+--------------+-------------+-------------+---------------+---------------+-----+---------+-------------------+-----------------------+----------------+------------------+-------------------------+--------------------+
|c422d6561ae540e40...|     Client services|         1|          RENT|      44000.0|           NH|          030xx|            USA|    G|       G1|       Not Verified|                

26/02/08 22:31:23 WARN HiveExternalCatalog: Couldn't find corresponding Hive SerDe for data source provider csv. Persisting data source table `spark_catalog`.`lending_club`.`customers` into Hive metastore in Spark SQL specific format, which is NOT compatible with Hive.


In [150]:
spark.sql("""
create external table lending_club.loans(
loan_id string, member_id string, loan_amount float, funded_amount float,
loan_term_years integer, interest_rate float, monthly_installment float, issue_date string,
loan_status string, loan_purpose string, loan_title string, ingest_date timestamp)
using csv location  '/Users/himanshuhegde/Desktop/Lending_club/data/cleaned/loans.csv'
options('header'='true', 'inferSchema'='false')
""")
spark.sql("select * from lending_club.loans").show(5)


+---------+--------------------+-----------+-------------+---------------+-------------+-------------------+----------+-----------+------------------+--------------------+--------------------+
|  loan_id|           member_id|loan_amount|funded_amount|loan_term_years|interest_rate|monthly_installment|issue_date|loan_status|      loan_purpose|          loan_title|         ingest_date|
+---------+--------------------+-----------+-------------+---------------+-------------+-------------------+----------+-----------+------------------+--------------------+--------------------+
|145499677|a703357afc7be3fe3...|    10000.0|      10000.0|              3|         8.19|             314.25|  Dec-2018| Fully Paid|debt_consolidation|  Debt consolidation|2026-02-08 13:14:...|
|144538467|a0c637c3df6764663...|     5000.0|       5000.0|              3|        15.02|             173.38|  Dec-2018|    Current|             other|               Other|2026-02-08 13:14:...|
|145515405|63571114d3a96e5bc...|   

26/02/08 22:31:26 WARN HiveExternalCatalog: Couldn't find corresponding Hive SerDe for data source provider csv. Persisting data source table `spark_catalog`.`lending_club`.`loans` into Hive metastore in Spark SQL specific format, which is NOT compatible with Hive.


In [151]:
spark.sql("""CREATE EXTERNAL TABLE lending_club.repayments(
    loan_id string, total_principal_received float, 
    total_interest_received float,total_late_fee_received float,
    total_payment_received float,last_payment_amount float,
    last_payment_date string,next_payment_date string,
    ingest_date timestamp)
    using csv location  '/Users/himanshuhegde/Desktop/Lending_club/data/cleaned/repayments.csv'
    options('header'='true', 'inferSchema'='false')
""")
spark.sql("select * from lending_club.repayments").show(5)


+---------+------------------------+-----------------------+-----------------------+----------------------+-------------------+-----------------+-----------------+--------------------+
|  loan_id|total_principal_received|total_interest_received|total_late_fee_received|total_payment_received|last_payment_amount|last_payment_date|next_payment_date|         ingest_date|
+---------+------------------------+-----------------------+-----------------------+----------------------+-------------------+-----------------+-----------------+--------------------+
|141581221|                 1055.81|                 2591.7|                    0.0|               3647.51|             709.23|         Mar-2019|         Mar-2019|2026-02-08 22:23:...|
|141506948|                 1252.75|                 306.04|                    0.0|               1558.79|             312.63|         Mar-2019|         Mar-2019|2026-02-08 22:23:...|
|141357400|                  626.37|                 354.96|               

26/02/08 22:31:28 WARN HiveExternalCatalog: Couldn't find corresponding Hive SerDe for data source provider csv. Persisting data source table `spark_catalog`.`lending_club`.`repayments` into Hive metastore in Spark SQL specific format, which is NOT compatible with Hive.


In [152]:
spark.sql("""CREATE EXTERNAL TABLE lending_club.delinquencies(
    member_id string, delinq_2yrs integer, delinq_amnt float,
    mths_since_last_delinq integer)
    using csv location  '/Users/himanshuhegde/Desktop/Lending_club/data/cleaned/delinquencies.csv'
    options('header'='true', 'inferSchema'='false')
""")
spark.sql("select * from lending_club.delinquencies").show(5)


+--------------------+-----------+-----------+----------------------+
|           member_id|delinq_2yrs|delinq_amnt|mths_since_last_delinq|
+--------------------+-----------+-----------+----------------------+
|0000036e9afe019a6...|          2|        0.0|                    15|
|00006cfe783695214...|          0|        0.0|                    58|
|000077b348e7220f6...|          1|        0.0|                    18|
|00012644aa138932f...|          2|        0.0|                    18|
|000128ed80bee9d2e...|          1|        0.0|                    23|
+--------------------+-----------+-----------+----------------------+
only showing top 5 rows


26/02/08 22:31:31 WARN HiveExternalCatalog: Couldn't find corresponding Hive SerDe for data source provider csv. Persisting data source table `spark_catalog`.`lending_club`.`delinquencies` into Hive metastore in Spark SQL specific format, which is NOT compatible with Hive.


In [148]:
spark.sql("""CREATE EXTERNAL TABLE lending_club.delinquencies_rec_enq(
    member_id string, pub_rec integer, pub_rec_bankruptcies integer, 
    inq_last_6mths integer)
    using csv location  '/Users/himanshuhegde/Desktop/Lending_club/data/cleaned/loans_del_detail_records_enq_df.csv'
    options('header'='true', 'inferSchema'='false')
    """)

spark.sql("select * from lending_club.delinquencies_rec_enq").show(5)

+--------------------+-------+--------------------+--------------+
|           member_id|pub_rec|pub_rec_bankruptcies|inq_last_6mths|
+--------------------+-------+--------------------+--------------+
|0000049d2d8e1de0e...|      0|                   0|             0|
|000004ef9774512ca...|      0|                   0|             1|
|00000a8865d7fcbe1...|      1|                   1|             3|
|00000e57abadb0ca9...|      0|                   0|             0|
|000037ccc3f6a992e...|      0|                   0|             0|
+--------------------+-------+--------------------+--------------+
only showing top 5 rows


26/02/08 22:31:08 WARN HiveExternalCatalog: Couldn't find corresponding Hive SerDe for data source provider csv. Persisting data source table `spark_catalog`.`lending_club`.`delinquencies_rec_enq` into Hive metastore in Spark SQL specific format, which is NOT compatible with Hive.


In [ ]:
spark.sql("""
create or replace view lending_club.customers_loan_v as select
l.loan_id,
c.member_id,
c.emp_title,
c.emp_length,
c.home_ownership,
c.annual_income,
c.address_state,
c.address_zipcode,
c.address_country,
c.grade,
c.sub_grade,
c.verification_status,
c.total_high_credit_limit,
c.application_type,
c.join_annual_income,
c.verification_status_joint,
l.loan_amount,
l.funded_amount,
l.loan_term_years,
l.interest_rate,
l.monthly_installment,
l.issue_date,
l.loan_status,
l.loan_purpose,
r.total_principal_received,
r.total_interest_received,
r.total_late_fee_received,
r.last_payment_date,
r.next_payment_date,
d.delinq_2yrs,
d.delinq_amnt,
d.mths_since_last_delinq,
e.pub_rec,
e.pub_rec_bankruptcies,
e.inq_last_6mths

FROM lending_club.customers c
LEFT JOIN lending_club.loans l on c.member_id = l.member_id
LEFT JOIN lending_club.loans_repayments r ON l.loan_id = r.loan_id
LEFT JOIN lending_club.delinquencies d ON c.member_id = d.member_id
LEFT JOIN lending_club.delinquencies_rec_enq e ON c.member_id = e.member_id
""")

spark.sql("select * from lending_club.customers_loan_v").show(5)


+---------+--------------------+--------------------+----------+--------------+-------------+-------------+---------------+---------------+-----+---------+-------------------+-----------------------+----------------+------------------+-------------------------+-----------+-------------+---------------+-------------+-------------------+----------+-----------+------------------+------------------------+-----------------------+-----------------------+-----------------+-----------------+-----------+-----------+----------------------+-------+--------------------+--------------+
|  loan_id|           member_id|           emp_title|emp_length|home_ownership|annual_income|address_state|address_zipcode|address_country|grade|sub_grade|verification_status|total_high_credit_limit|application_type|join_annual_income|verification_status_joint|loan_amount|funded_amount|loan_term_years|interest_rate|monthly_installment|issue_date|loan_status|      loan_purpose|total_principal_received|total_interest_r

In [157]:
spark.sql("drop table if exists lending_club.customers_loan_t")

spark.sql("""
create table lending_club.customers_loan_t as select
l.loan_id,
c.member_id,
c.emp_title,
c.emp_length,
c.home_ownership,
c.annual_income,
c.address_state,
c.address_zipcode,
c.address_country,
c.grade,
c.sub_grade,
c.verification_status,
c.total_high_credit_limit,
c.application_type,
c.join_annual_income,
c.verification_status_joint,
l.loan_amount,
l.funded_amount,
l.loan_term_years,
l.interest_rate,
l.monthly_installment,
l.issue_date,
l.loan_status,
l.loan_purpose,
r.total_principal_received,
r.total_interest_received,
r.total_late_fee_received,
r.last_payment_date,
r.next_payment_date,
d.delinq_2yrs,
d.delinq_amnt,
d.mths_since_last_delinq,
e.pub_rec,
e.pub_rec_bankruptcies,
e.inq_last_6mths

FROM lending_club.customers c
LEFT JOIN lending_club.loans l on c.member_id = l.member_id
LEFT JOIN lending_club.loans_repayments r ON l.loan_id = r.loan_id
LEFT JOIN lending_club.delinquencies d ON c.member_id = d.member_id
LEFT JOIN lending_club.delinquencies_rec_enq e ON c.member_id = e.member_id
""")

spark.sql("select * from lending_club.customers_loan_t").show(5)


+---------+--------------------+--------------------+----------+--------------+-------------+-------------+---------------+---------------+-----+---------+-------------------+-----------------------+----------------+------------------+-------------------------+-----------+-------------+---------------+-------------+-------------------+----------+-----------+------------------+------------------------+-----------------------+-----------------------+-----------------+-----------------+-----------+-----------+----------------------+-------+--------------------+--------------+
|  loan_id|           member_id|           emp_title|emp_length|home_ownership|annual_income|address_state|address_zipcode|address_country|grade|sub_grade|verification_status|total_high_credit_limit|application_type|join_annual_income|verification_status_joint|loan_amount|funded_amount|loan_term_years|interest_rate|monthly_installment|issue_date|loan_status|      loan_purpose|total_principal_received|total_interest_r

Remove duplicate member ids

In [ ]:
spark.sql("""select member_id, count(*) as total 
from lending_club.customers
group by member_id order by total desc
""").show()

In [ ]:
spark.sql("""select member_id, count(*) as total 
from lending_club.loans
group by member_id order by total desc
""").show()

In [ ]:
spark.sql("""select member_id, count(*) as total 
from lending_club.repayments
group by member_id order by total desc
""").show()

In [ ]:
spark.sql("""select member_id, count(*) as total 
from lending_club.loans_defaulters_delinq
group by member_id order by total desc
""").show()

In [ ]:
spark.sql("""select member_id, count(*) as total 
from lending_club.loans_defaulters_detail_rec_enq
group by member_id order by total desc
""").show()

In [ ]:
bad_data_customer_df = spark.sql("""select member_id from(select member_id, count(*)
as total from lending_club.customers
group by member_id having total > 1)""")

In [ ]:
bad_data_customer_df.show()

In [ ]:
bad_data_customer_df.count()

In [ ]:
bad_data_loans_defaulters_delinq_df = spark.sql("""select member_id from(select member_id, count(*)
as total from lending_club.loans_defaulters_delinq
group by member_id having total > 1)""")

In [ ]:
bad_data_loans_defaulters_delinq_df.count()

In [ ]:
bad_data_loans_defaulters_detail_rec_enq_df = spark.sql("""
    SELECT member_id 
    FROM (SELECT member_id, count(*)
            AS total from lending_club.loans_defaulters_detail_rec_enq
            GROUP BY member_id having total > 1)
    """)

bad_data_loans_defaulters_detail_rec_enq_df.count()

In [ ]:
bad_data_customer_df.repartition(1).write \
.format("csv") \
.option("header", True) \
.mode("overwrite") \
.option("path", "../data/bad/bad_data_customers.csv") \
.save()

In [ ]:
bad_data_loans_defaulters_delinq_df.repartition(1).write \
.format("csv") \
.option("header", True) \
.mode("overwrite") \
.option("path", "../data/bad/bad_data_loans_defaulters_delinq.csv") \
.save()

In [ ]:
bad_customer_data_df = bad_data_customer_df.select("member_id") \
.union(bad_data_loans_defaulters_delinq_df.select("member_id")) \
.union(bad_data_loans_defaulters_detail_rec_enq_df.select("member_id"))

In [ ]:
bad_customer_data_final_df = bad_customer_data_df.distinct()

In [ ]:
bad_customer_data_final_df.count()
bad_customer_data_final_df.show()

In [ ]:
bad_customer_data_final_df.repartition(1).write \
.format("csv") \
.option("header", True) \
.mode("overwrite") \
.option("path", "../data/bad/bad_customer_data_final.csv") \
.save()

In [ ]:
bad_customer_data_final_df.createOrReplaceTempView("bad_data_customer")

In [ ]:
customers_df = spark.sql("""select * from lending_club.customers
where member_id NOT IN (select member_id from bad_data_customer)
""")


In [ ]:
customers_df.show()

In [ ]:
customers_df.write \
.format("parquet") \
.mode("overwrite") \
.option("path", "../data/cleaned_new/customers_parquet") \
.save()

In [ ]:
temp = spark.read.parquet("../data/cleaned_new/customers_parquet")
temp.show()

In [ ]:
loans_defaulters_delinq_df = spark.sql("""select * from lending_club.loans_defaulters_delinq
where member_id NOT IN (select member_id from bad_data_customer)
""")

In [ ]:
loans_defaulters_delinq_df.show()

In [ ]:
loans_defaulters_delinq_df.write \
.format("parquet") \
.mode("overwrite") \
.option("path", "../data/cleaned_new/loans_defaulters_delinq_parquet") \
.save()

In [ ]:
temp = spark.read.parquet("../data/cleaned_new/loans_defaulters_delinq_parquet")
temp.show()

In [ ]:
loans_defaulters_detail_rec_enq_df = spark.sql("""select * from lending_club.loans_defaulters_detail_rec_enq
where member_id NOT IN (select member_id from bad_data_customer)
""")

In [ ]:
loans_defaulters_detail_rec_enq_df.write \
.format("parquet") \
.mode("overwrite") \
.option("path", "../data/cleaned_new/loans_defaulters_detail_rec_enq_parquet") \
.save()

In [ ]:
temp = spark.read.parquet("../data/cleaned_new/loans_defaulters_detail_rec_enq_parquet")
temp.show()

In [ ]:
spark.sql("DROP TABLE IF EXISTS lending_club.loans_defaulters_detail_rec_enq_new")

In [ ]:
spark.sql("""
create EXTERNAL TABLE lending_club.customers_new(
    member_id string, emp_title string, emp_length int, home_ownership string, 
    annual_income float, address_state string, address_zipcode string, 
    address_country string, grade string, sub_grade string, verification_status string,
    total_high_credit_limit float, application_type string, 
    join_annual_income float, verification_status_joint string, ingest_date timestamp)
    stored as parquet
    LOCATION '/Users/himanshuhegde/Desktop/Lending_club/data/cleaned_new/customers_parquet'
    
""")

In [ ]:
spark.sql("select * from lending_club.customers_new").show()

In [ ]:
spark.sql("""
create EXTERNAL TABLE lending_club.loans_defaulters_delinq_new(
    member_id string,delinq_2yrs integer, delinq_amnt float, mths_since_last_delinq integer)
    stored as parquet
    LOCATION '/Users/himanshuhegde/Desktop/Lending_club/data/cleaned_new/loans_defaulters_delinq_parquet'
""")

In [ ]:
spark.sql("select * from lending_club.loans_defaulters_delinq_new").show()

In [ ]:
spark.sql("""
create EXTERNAL TABLE lending_club.loans_defaulters_detail_rec_enq_new(
    member_id string, pub_rec integer, pub_rec_bankruptcies integer, inq_last_6mths integer)
    stored as parquet
    LOCATION '/Users/himanshuhegde/Desktop/Lending_club/data/cleaned_new/loans_defaulters_detail_rec_enq_parquet'
""")

In [ ]:
spark.sql("select * from lending_club.loans_defaulters_detail_rec_enq_new").show()

In [ ]:
spark.sql("""select member_id, count(*) as total 
from lending_club.customers_new
group by member_id order by total desc""").show()

### 6. Calculating credit scores

In [158]:
credit_score_tiers = {
    "unacceptable": 0,
    "very_bad": 100,
    "bad": 250,
    "good": 500,
    "very_good": 650,
    "excellent": 800
}

In [159]:
#Reading bad data for excluding them later on
bad_customer_data_final_df = spark.read \
.format("csv") \
.option("header", True) \
.option("inferSchema", True) \
.load("../data/bad/bad_customer_data_final.csv")


In [ ]:
bad_customer_data_final_df.createOrReplaceTempView("bad_data_customer")

In [160]:
ph_df = spark.sql(f"""
SELECT
    c.member_id,
    CASE
        WHEN p.last_payment_amount >= (c.monthly_installment * 0.5)
             AND p.last_payment_amount < c.monthly_installment
            THEN {credit_score_tiers["very_bad"]}
        WHEN p.last_payment_amount = c.monthly_installment
            THEN {credit_score_tiers["good"]}
        WHEN p.last_payment_amount > c.monthly_installment
             AND p.last_payment_amount <= (c.monthly_installment * 1.5)
            THEN {credit_score_tiers["very_good"]}
        WHEN p.last_payment_amount > (c.monthly_installment * 1.5)
            THEN {credit_score_tiers["excellent"]}
        ELSE {credit_score_tiers["unacceptable"]}
    END AS last_payment_pts,

    CASE
        WHEN p.total_payment_received >= (c.funded_amount * 0.5)
            THEN {credit_score_tiers["very_good"]}
        WHEN p.total_payment_received < (c.funded_amount * 0.5)
             AND p.total_payment_received > 0
            THEN {credit_score_tiers["good"]}
        WHEN p.total_payment_received = 0
             OR p.total_payment_received IS NULL
            THEN {credit_score_tiers["unacceptable"]}
    END AS total_payment_pts

FROM lending_club.loans_repayments p
JOIN lending_club.loans c
    ON c.loan_id = p.loan_id
""")

#WHERE c.member_id NOT IN (SELECT member_id FROM bad_data_customer)

In [161]:
ph_df.createOrReplaceTempView("ph_pts")

In [162]:
spark.sql("select * from ph_pts").show()

+--------------------+----------------+-----------------+
|           member_id|last_payment_pts|total_payment_pts|
+--------------------+----------------+-----------------+
|00720193212c0cb6f...|             800|              650|
|b1865c807a867bc00...|             800|              650|
|8b161e0f538f5ba02...|             500|              650|
|480514c1d6b3cf95a...|             500|              650|
|3dae0d924bcb74693...|             500|              650|
|15d328ff0703d79de...|             500|              650|
|18e6d99ee96fca904...|             500|              650|
|9f3cdf9809842d306...|             500|              650|
|2e89013db4b84924d...|             800|              650|
|d7095837304e24285...|             800|              650|
|26add02fec6758254...|             500|              650|
|6d34b3bfafc54a21f...|             500|              650|
|e6dd98c595acc43b9...|             800|              650|
|8c6a3bbfb669563a9...|             500|              650|
|0c8f1155b6825

In [ ]:
# calculation criterion 2 - loan defaulters history ldh

In [163]:
ldh_ph_df = spark.sql(f"""
    select p.*,
    CASE 
    WHEN d.delinq_2yrs = 0 
        THEN {credit_score_tiers["excellent"]} 
    WHEN d.delinq_2yrs BETWEEN 1 AND 2 
        THEN {credit_score_tiers["bad"]} 
    WHEN d.delinq_2yrs BETWEEN 3 AND 5 
        THEN {credit_score_tiers["very_bad"]} 
    WHEN d.delinq_2yrs > 5 OR d.delinq_2yrs IS NULL 
        THEN {credit_score_tiers["unacceptable"]} 
    END AS delinq_pts, 
    CASE 
    WHEN l.pub_rec = 0 
        THEN {credit_score_tiers["excellent"]} 
    WHEN l.pub_rec BETWEEN 1 AND 2 
        THEN {credit_score_tiers["bad"]} 
    WHEN l.pub_rec BETWEEN 3 AND 5 
        THEN {credit_score_tiers["very_bad"]} 
    WHEN l.pub_rec > 5 OR l.pub_rec IS NULL 
        THEN {credit_score_tiers["very_bad"]} 
    END AS public_records_pts, 
    CASE 
    WHEN l.pub_rec_bankruptcies = 0 
        THEN {credit_score_tiers["excellent"]} 
    WHEN l.pub_rec_bankruptcies BETWEEN 1 AND 2 
        THEN {credit_score_tiers["bad"]} 
    WHEN l.pub_rec_bankruptcies BETWEEN 3 AND 5 
        THEN {credit_score_tiers["very_bad"]} 
    WHEN l.pub_rec_bankruptcies > 5 OR l.pub_rec_bankruptcies IS NULL 
        THEN {credit_score_tiers["very_bad"]}
    END as public_bankruptcies_pts, 
    CASE 
    WHEN l.inq_last_6mths = 0 
        THEN {credit_score_tiers["excellent"]} 
    WHEN l.inq_last_6mths BETWEEN 1 AND 2 
        THEN {credit_score_tiers["bad"]} 
    WHEN l.inq_last_6mths BETWEEN 3 AND 5 
        THEN {credit_score_tiers["very_bad"]} 
    WHEN l.inq_last_6mths > 5 OR l.inq_last_6mths IS NULL 
        THEN {credit_score_tiers["unacceptable"]} 
    END AS enq_pts 
    FROM lending_club.loans_defaulters_detail_rec_enq_new l 
    INNER JOIN lending_club.loans_defaulters_delinq_new d ON d.member_id = l.member_id  
    INNER JOIN ph_pts p ON p.member_id = l.member_id
    """)
#where l.member_id NOT IN (select member_id from bad_data_customer)

In [164]:
ldh_ph_df.createOrReplaceTempView("ldh_ph_pts")

In [165]:
spark.sql("select * from ldh_ph_pts").show()

+--------------------+----------------+-----------------+----------+------------------+-----------------------+-------+
|           member_id|last_payment_pts|total_payment_pts|delinq_pts|public_records_pts|public_bankruptcies_pts|enq_pts|
+--------------------+----------------+-----------------+----------+------------------+-----------------------+-------+
|0000036e9afe019a6...|             800|              650|       250|               800|                    800|    100|
|000128ed80bee9d2e...|             500|              500|       250|               800|                    800|    800|
|000152208b3e77b5b...|             500|              650|       800|               800|                    800|    800|
|000170b4ccb292792...|             800|              650|       250|               800|                    800|    250|
|0001cfa200f7480b9...|             500|              650|       250|               250|                    250|    250|
|00024adf1230710bd...|             500| 

In [166]:
fh_ldh_ph_df = spark.sql(f"""select ldef.*, 
    CASE 
    WHEN LOWER(l.loan_status) LIKE '%fully paid%'
        THEN {credit_score_tiers["excellent"]} 
    WHEN LOWER(l.loan_status) LIKE '%current%' 
        THEN {credit_score_tiers["good"]} 
    WHEN LOWER(l.loan_status) LIKE '%in grace period%' 
        THEN {credit_score_tiers["bad"]} 
    WHEN LOWER(l.loan_status) LIKE '%late (16-30 days)%' OR LOWER(l.loan_status) LIKE '%late (31-120 days)%' 
        THEN {credit_score_tiers["very_bad"]} 
    WHEN LOWER(l.loan_status) LIKE '%charged off%' 
        THEN {credit_score_tiers["unacceptable"]}
    else {credit_score_tiers["unacceptable"]} 
    END AS loan_status_pts, 
    CASE 
    WHEN LOWER(a.home_ownership) LIKE '%own' 
        THEN {credit_score_tiers["excellent"]} 
    WHEN LOWER(a.home_ownership) LIKE '%rent' 
        THEN {credit_score_tiers["good"]} 
    WHEN LOWER(a.home_ownership) LIKE '%mortgage' 
        THEN {credit_score_tiers["bad"]} 
    WHEN LOWER(a.home_ownership) LIKE '%any' OR LOWER(a.home_ownership) IS NULL 
        THEN {credit_score_tiers["very_bad"]} 
    END AS home_pts, 
    CASE 
    WHEN l.funded_amount <= (a.total_high_credit_limit * 0.10) 
        THEN {credit_score_tiers["excellent"]} 
    WHEN l.funded_amount > (a.total_high_credit_limit * 0.10) AND l.funded_amount <= (a.total_high_credit_limit * 0.20) 
        THEN {credit_score_tiers["very_good"]} 
    WHEN l.funded_amount > (a.total_high_credit_limit * 0.20) AND l.funded_amount <= (a.total_high_credit_limit * 0.30) 
        THEN {credit_score_tiers["good"]} 
    WHEN l.funded_amount > (a.total_high_credit_limit * 0.30) AND l.funded_amount <= (a.total_high_credit_limit * 0.50) 
        THEN {credit_score_tiers["bad"]} 
    WHEN l.funded_amount > (a.total_high_credit_limit * 0.50) AND l.funded_amount <= (a.total_high_credit_limit * 0.70) 
        THEN {credit_score_tiers["very_bad"]} 
    WHEN l.funded_amount > (a.total_high_credit_limit * 0.70) 
        THEN {credit_score_tiers["unacceptable"]}
    else {credit_score_tiers["unacceptable"]} 
    END AS credit_limit_pts, 
    CASE 
    WHEN (a.grade) = 'A' and (a.sub_grade)='A1' 
        THEN {credit_score_tiers["excellent"]} 
    WHEN (a.grade) = 'A' and (a.sub_grade)='A2' 
        THEN ({credit_score_tiers["excellent"]} * 0.95) 
    WHEN (a.grade) = 'A' and (a.sub_grade)='A3' 
        THEN ({credit_score_tiers["excellent"]} * 0.90) 
    WHEN (a.grade) = 'A' and (a.sub_grade)='A4' 
        THEN ({credit_score_tiers["excellent"]} * 0.85) 
    WHEN (a.grade) = 'A' and (a.sub_grade)='A5' 
        THEN ({credit_score_tiers["excellent"]} * 0.80) 
    WHEN (a.grade) = 'B' and (a.sub_grade)='B1' 
        THEN {credit_score_tiers["very_good"]} 
    WHEN (a.grade) = 'B' and (a.sub_grade)='B2' 
        THEN ({credit_score_tiers["very_good"]} * 0.95) 
    WHEN (a.grade) = 'B' and (a.sub_grade)='B3' 
        THEN ({credit_score_tiers["very_good"]} * 0.90) 
    WHEN (a.grade) = 'B' and (a.sub_grade)='B4' 
        THEN ({credit_score_tiers["very_good"]} * 0.85) 
    WHEN (a.grade) = 'B' and (a.sub_grade)='B5' 
        THEN ({credit_score_tiers["very_good"]} * 0.80) 
    WHEN (a.grade) = 'C' and (a.sub_grade)='C1' 
        THEN {credit_score_tiers["good"]} 
    WHEN (a.grade) = 'C' and (a.sub_grade)='C2' 
        THEN ({credit_score_tiers["good"]} * 0.95) 
    WHEN (a.grade) = 'C' and (a.sub_grade)='C3' 
        THEN ({credit_score_tiers["good"]} * 0.90) 
    WHEN (a.grade) = 'C' and (a.sub_grade)='C4' 
        THEN ({credit_score_tiers["good"]} * 0.85) 
    WHEN (a.grade) = 'C' and (a.sub_grade)='C5' 
        THEN ({credit_score_tiers["good"]} * 0.80) 
    WHEN (a.grade) = 'D' and (a.sub_grade)='D1' THEN ({credit_score_tiers["bad"]}) 
    WHEN (a.grade) = 'D' and (a.sub_grade)='D2' THEN ({credit_score_tiers["bad"]} * 0.95) 
    WHEN (a.grade) = 'D' and (a.sub_grade)='D3' THEN ({credit_score_tiers["bad"]} * 0.90) 
    WHEN (a.grade) = 'D' and (a.sub_grade)='D4' THEN ({credit_score_tiers["bad"]} * 0.85) 
    WHEN (a.grade) = 'D' and (a.sub_grade)='D5' THEN ({credit_score_tiers["bad"]} * 0.80) 
    WHEN (a.grade) = 'E' and (a.sub_grade)='E1' THEN {credit_score_tiers["very_bad"]} 
    WHEN (a.grade) = 'E' and (a.sub_grade)='E2' THEN ({credit_score_tiers["very_bad"]} * 0.95) 
    WHEN (a.grade) = 'E' and (a.sub_grade)='E3' THEN ({credit_score_tiers["very_bad"]} * 0.90) 
    WHEN (a.grade) = 'E' and (a.sub_grade)='E4' THEN ({credit_score_tiers["very_bad"]} * 0.85) 
    WHEN (a.grade) = 'E' and (a.sub_grade)='E5' THEN ({credit_score_tiers["very_bad"]} * 0.80) 
    WHEN (a.grade) in ('F', 'G') THEN {credit_score_tiers["unacceptable"]} 
    END AS grade_pts 
    FROM ldh_ph_pts ldef 
    INNER JOIN lending_club.loans l ON ldef.member_id = l.member_id 
    INNER JOIN lending_club.customers_new a ON a.member_id = ldef.member_id
   """) 
#where ldef.member_id NOT IN (select member_id from bad_data_customer)


In [167]:
fh_ldh_ph_df.createOrReplaceTempView("fh_ldh_ph_pts")
spark.sql("select * from fh_ldh_ph_pts").show()


+--------------------+----------------+-----------------+----------+------------------+-----------------------+-------+---------------+--------+----------------+---------+
|           member_id|last_payment_pts|total_payment_pts|delinq_pts|public_records_pts|public_bankruptcies_pts|enq_pts|loan_status_pts|home_pts|credit_limit_pts|grade_pts|
+--------------------+----------------+-----------------+----------+------------------+-----------------------+-------+---------------+--------+----------------+---------+
|0000036e9afe019a6...|             800|              650|       250|               800|                    800|    100|            800|     250|             800|   500.00|
|000152208b3e77b5b...|             500|              650|       800|               800|                    800|    800|            500|     500|             650|   250.00|
|000170b4ccb292792...|             800|              650|       250|               800|                    800|    250|            800|     

In [168]:
# Final Credit Score Calculation
# 1. Payment History = 20%
# 2. Loan Defaults = 45%
# 3. Financial Health = 35%

In [169]:
credit_score = spark.sql("""SELECT member_id, 
    ((last_payment_pts+total_payment_pts)*0.20) as payment_history_pts, 
    ((delinq_pts + public_records_pts + public_bankruptcies_pts + enq_pts) * 0.45) as defaulters_history_pts, 
    ((loan_status_pts + home_pts + credit_limit_pts + grade_pts)*0.35) as financial_health_pts 
    FROM fh_ldh_ph_pts""")

In [170]:
credit_score.show()

+--------------------+-------------------+----------------------+--------------------+
|           member_id|payment_history_pts|defaulters_history_pts|financial_health_pts|
+--------------------+-------------------+----------------------+--------------------+
|0000036e9afe019a6...|             290.00|                877.50|            822.5000|
|000152208b3e77b5b...|             230.00|               1440.00|            665.0000|
|000170b4ccb292792...|             290.00|                945.00|            852.2500|
|0001cfa200f7480b9...|             230.00|                450.00|            665.0000|
|00024adf1230710bd...|             200.00|               1192.50|            682.5000|
|00026136ec721b938...|             200.00|                945.00|            653.6250|
|00030e831c078f92a...|             230.00|               1440.00|            595.8750|
|0004656412a18b9c1...|             290.00|               1192.50|            988.7500|
|0004f290201c29d93...|             200.00| 

In [171]:
final_credit_score = credit_score.withColumn('credit_score', credit_score.payment_history_pts + credit_score.defaulters_history_pts + credit_score.financial_health_pts)

In [172]:
final_credit_score.createOrReplaceTempView("credit_score_eval")

In [173]:
final_credit_score.show()

+--------------------+-------------------+----------------------+--------------------+------------+
|           member_id|payment_history_pts|defaulters_history_pts|financial_health_pts|credit_score|
+--------------------+-------------------+----------------------+--------------------+------------+
|0000036e9afe019a6...|             290.00|                877.50|            822.5000|   1990.0000|
|000152208b3e77b5b...|             230.00|               1440.00|            665.0000|   2335.0000|
|000170b4ccb292792...|             290.00|                945.00|            852.2500|   2087.2500|
|0001cfa200f7480b9...|             230.00|                450.00|            665.0000|   1345.0000|
|00024adf1230710bd...|             200.00|               1192.50|            682.5000|   2075.0000|
|00026136ec721b938...|             200.00|                945.00|            653.6250|   1798.6250|
|00030e831c078f92a...|             230.00|               1440.00|            595.8750|   2265.8750|


In [174]:
final_credit_score.write \
.format("parquet") \
.mode("overwrite") \
.option("path", "../data/processed/credit_score") \
.save()